# Manual validation of DeepSeek annotations: accusations & role (identity) claims

This notebook draws a **random sample of 30 items** for each of the two DeepSeek-annotated
tasks in the lai2023 (ONUW) corpus and walks you through them one at a time so you can judge
whether DeepSeek's extraction was correct:

- **Accusation targets** (`accusation_transcripts/acc_targets/**/*.json`) — for each flagged
  line, DeepSeek extracted who the *accuser* was and any *relations* (`werewolf`/`deception`
  accusation type, who was *accused*, and the evidence quote).
- **Identity claims** (`identity_claim_transcripts/ic_targets/**/*.json`) — for each flagged
  line, DeepSeek extracted the *role(s)* a player claimed to be (e.g. Seer, Werewolf, Robber).

For each sampled item you'll see the surrounding transcript context (target line marked `>>>`)
and DeepSeek's prediction, then answer:

- `c` = correct, `i` = incorrect, `p` = partially correct, `s` = skip for now, `q` = stop the loop
- if not fully correct, you'll be asked what the right answer should have been (free text)
- an optional free-text note

Progress is saved to CSV after every answer, so it's safe to stop and resume anytime — just
re-run the notebook from the top; the same 30-item sample is reloaded (not re-randomized) and
only unreviewed rows are shown.

Output files (created on first run):
- `data/processed/lai2023/manual_validation/accusation_manual_review.csv`
- `data/processed/lai2023/manual_validation/identity_claim_manual_review.csv`


In [1]:
import json
import csv
import random
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# ---- adjust these if your paths differ ----
BASE = Path("C:/Users/annab/Documents/GitHub/masters_thesis_sdg")
DATA_ROOT = BASE / "data" / "processed" / "lai2023"
VALIDATION_ROOT = DATA_ROOT / "manual_validation"
RANDOM_SEED = 42
CONTEXT_BEFORE = 10
CONTEXT_AFTER = 10
# --------------------------------------------


def parse_transcript_line(raw):
    """Parse a '[12] Speaker: text' transcript line into (line_number, text)."""
    if not raw.startswith("["):
        return None
    end = raw.find("]")
    if end == -1:
        return None
    num_str = raw[1:end]
    if not num_str.isdigit():
        return None
    text = raw[end + 1:].strip()
    for marker in ("<<ACCUSATION_TO_RESOLVE>>", "<<IDENTITY_CLAIM_TO_RESOLVE>>"):
        text = text.replace(marker, "")
    return int(num_str), text.strip()


def load_transcript_lines(txt_path):
    lines = {}
    for raw in txt_path.read_text(encoding="utf-8").splitlines():
        parsed = parse_transcript_line(raw)
        if parsed:
            num, text = parsed
            lines[num] = text
    return lines


def get_context(lines, target, before=CONTEXT_BEFORE, after=CONTEXT_AFTER):
    nums = sorted(n for n in lines if target - before <= n <= target + after)
    return [(n, lines[n]) for n in nums]


def format_relations(relations):
    if not relations:
        return "(none)"
    parts = []
    for r in relations:
        accused = ", ".join(r.get("accused", [])) or "?"
        parts.append(f"{r.get('type', '?')} -> {accused} (evidence: {r.get('evidence', '')})")
    return " | ".join(parts)


def build_item_index(json_root):
    """Flatten every {metadata, item} pair across all per-game JSON output files."""
    index = []
    for json_path in sorted(json_root.rglob("*.json")):
        try:
            record = json.loads(json_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            continue
        meta = record.get("metadata", {})
        for item in record.get("items", []):
            index.append({"json_path": json_path, "metadata": meta, "item": item})
    return index


def save_rows(rows, csv_path):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys()) if rows else []
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_rows(csv_path):
    with csv_path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def build_or_load_sample(task_key, config):
    """Load the existing review CSV if present (keeps the sample + progress stable
    across reruns), otherwise draw a fresh random sample and write it out."""
    csv_path = config["review_csv"]
    if csv_path.exists():
        rows = load_rows(csv_path)
        print(f"[{task_key}] loaded existing sample of {len(rows)} items from {csv_path.name}")
        return rows

    index = build_item_index(config["json_root"])
    print(f"[{task_key}] {len(index)} total annotated items found across corpus")
    random.seed(RANDOM_SEED)
    sample = random.sample(index, min(config["n_samples"], len(index)))

    rows = []
    for entry in sample:
        meta = entry["metadata"]
        item = entry["item"]
        json_path = entry["json_path"]
        txt_rel = meta.get("source_file", "").replace(chr(92), "/")
        txt_path = config["txt_root"] / txt_rel
        line_no = item.get("line_number")
        target_text = ""
        if txt_path.exists():
            target_text = load_transcript_lines(txt_path).get(line_no, "")

        row = {
            "item_id": f"{meta.get('source')}|{meta.get('session')}|{meta.get('game')}|{line_no}",
            "source": meta.get("source", ""),
            "session": meta.get("session", ""),
            "game": meta.get("game", ""),
            "output_file": str(json_path.relative_to(config["json_root"])),
            "txt_file": txt_rel,
            "line_number": line_no,
            "target_line_text": target_text,
        }
        row.update(config["predicted_fields"](item))
        row.update({"human_judgment": "", "human_correction": "", "notes": "", "reviewed_at": ""})
        rows.append(row)

    save_rows(rows, csv_path)
    print(f"[{task_key}] created new sample of {len(rows)} items -> {csv_path}")
    return rows


def review_loop(task_key, config, rows):
    csv_path = config["review_csv"]
    pending = [r for r in rows if r["human_judgment"] == ""]
    print(f"[{task_key}] {len(rows)} total, {len(pending)} pending review.")

    for row in pending:
        print("=" * 70)
        print(f"[{task_key}] {row['source']} / {row['game']}  (line {row['line_number']})")
        txt_path = config["txt_root"] / row["txt_file"]
        target_line = int(row["line_number"])
        if txt_path.exists():
            context = get_context(load_transcript_lines(txt_path), target_line)
            for n, text in context:
                marker = ">>>" if n == target_line else "   "
                print(f"{marker} [{n}] {text}")
        else:
            print(f"  (transcript not found: {txt_path})")
        print("-" * 70)
        for key in row:
            if key.endswith("_predicted"):
                print(f"{key:22s}: {row[key]}")
        print("-" * 70)

        answer = input("Judgment: [c]orrect / [i]ncorrect / [p]artial / [s]kip / [q]uit: ").strip().lower()
        if answer == "q":
            print("Stopping. Progress so far is saved.")
            break
        if answer in ("", "s"):
            continue
        judgment = {"c": "correct", "i": "incorrect", "p": "partial"}.get(answer)
        if judgment is None:
            print("  (unrecognized input -- treated as skip)")
            continue

        row["human_judgment"] = judgment
        if judgment != "correct":
            row["human_correction"] = input("What should it be? (Enter to skip): ").strip()
        row["notes"] = input("Notes (Enter to skip): ").strip()
        row["reviewed_at"] = datetime.now(timezone.utc).isoformat()
        save_rows(rows, csv_path)

    remaining = [r for r in rows if r["human_judgment"] == ""]
    print("=" * 70)
    print(f"[{task_key}] done for now, {len(remaining)} still pending.")


def summarize(task_key, rows):
    judged = [r for r in rows if r["human_judgment"]]
    n = len(judged)
    print(f"[{task_key}] {n}/{len(rows)} reviewed")
    if n == 0:
        return
    counts = Counter(r["human_judgment"] for r in judged)
    for k in ("correct", "partial", "incorrect"):
        print(f"  {k:10s}: {counts.get(k, 0)}")
    accuracy = counts.get("correct", 0) / n
    print(f"  accuracy (correct / reviewed): {accuracy:.1%}")


In [2]:
TASKS = {
    "accusation": {
        "json_root": DATA_ROOT / "accusation_transcripts" / "acc_targets",
        "txt_root": DATA_ROOT / "accusation_transcripts" / "ready_for_annotation",
        "review_csv": VALIDATION_ROOT / "accusation_manual_review.csv",
        "n_samples": 30,
        "predicted_fields": lambda item: {
            "accuser_predicted": item.get("accuser", ""),
            "relations_predicted": format_relations(item.get("relations", [])),
        },
    },
    "identity_claim": {
        "json_root": DATA_ROOT / "identity_claim_transcripts" / "ic_targets",
        "txt_root": DATA_ROOT / "identity_claim_transcripts" / "ready_for_annotation",
        "review_csv": VALIDATION_ROOT / "identity_claim_manual_review.csv",
        "n_samples": 30,
        "predicted_fields": lambda item: {
            "claimed_roles_predicted": ", ".join(item.get("claimed_roles", [])) or "(none)",
            "evidence_predicted": item.get("evidence", ""),
        },
    },
}


## Part 1 — Accusation targets (30 items)

Run the two cells below. The first draws/loads the sample; the second runs the interactive
review loop. Re-run the review cell to pick up where you left off (it re-reads `rows_accusation`,
so if you restart the kernel, re-run the build cell first — it will reload from CSV, not
re-sample).


In [3]:
rows_accusation = build_or_load_sample("accusation", TASKS["accusation"])


[accusation] 3505 total annotated items found across corpus
[accusation] created new sample of 30 items -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\manual_validation\accusation_manual_review.csv


In [4]:
review_loop("accusation", TASKS["accusation"], rows_accusation)


[accusation] 30 total, 30 pending review.
[accusation] Youtube / Game2  (line 73)
    [63] James: I was just hoping... That's the only information I'd be able to give is that, I was the troublemaker, and Paul is not, so-
    [64] Mitchell: If he's one of these three, I mean, It's 60% chance he's a werewolf.
    [65] James: Let's fucking just kill him.
    [66] Mitchell: We'll slow down.
    [67] Justin: We'll slow down.
    [68] James: Damn.
    [69] Mitchell: Now, let's look at everyone else's stories. I mean. This is all of us are constantly-
    [70] Justin: Yes. All of us are unconfirmed as well.
    [71] Mitchell: Yeah. There are no confirming-
    [72] James: Laura, can you confirm your story?
>>> [73] Mitchell: He hasn't spoken. So, I do highly suspect he is one of the three.
    [74] James: I suspect he's a tanner.
    [75] Justin: I think he's a tanner.
    [76] James: Let's vote left. Let's do it.
    [77] Justin: I don't think James is a tanner. Because, he's saying vote lef

## Part 2 — Identity (role) claims (30 items)

Same pattern: build/load the sample, then review. Independent of Part 1 — you can do the two
parts in any order or interleave sessions.


In [3]:
rows_identity_claim = build_or_load_sample("identity_claim", TASKS["identity_claim"])


[identity_claim] 1359 total annotated items found across corpus
[identity_claim] created new sample of 30 items -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\manual_validation\identity_claim_manual_review.csv


In [5]:
review_loop("identity_claim", TASKS["identity_claim"], rows_identity_claim)


[identity_claim] 30 total, 15 pending review.
[identity_claim] Youtube / Game3  (line 50)
    [40] Paul: Not, like all Canadian money.
    [41] Mitchell: Yeah like you're a loonies.
    [42] Paul: Yeah like a loonie.
    [43] Mitchell: You have a paper loonie.
    [44] Mitchell: You have your paper loonies.
    [45] Justin: Paper loonies.
    [46] Justin: My 50 loonie denouement James, can we hear where you are?
    [47] Mitchell: It cost 50 loonies.
    [48] Paul: It's a game and it's just like a number of loonies
    [49] Justin: Are you the hunter? Is that why you're so confident right now?
>>> [50] James: No. Some villager.
    [51] Mitchell: A villager.
    [52] James: Oh no.
    [53] Paul: Yeah.
    [54] Justin: Yeah, we need to hear from you too.
    [55] James: Well, I can confirm Mitch's story cause if you woke up, he woke up as that.
    [56] Paul: Here's the thing. If you were a werewolf, you're now the tanner. So then tell me if your werewolf is in your favor both ways.
   

## Summary

Quick accuracy readout for whatever has been reviewed so far in each part (safe to run at any
point, including partway through).


In [ ]:
summarize("accusation", rows_accusation)
print()
summarize("identity_claim", rows_identity_claim)
